# HealthBridge — 02: Multi-Model Training & Comparison


In [ ]:
import sys, json
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style='darkgrid')
ROOT = Path('..')


## 1. Run training (if not already done)

In [ ]:
import subprocess
result = subprocess.run(['python', '../src/train.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)


## 2. Load and display training summary

In [ ]:
with open(ROOT / 'reports/training_summary.json') as f:
    summary = json.load(f)

print(f"Best model: {summary['best_model']}")
print(f"N classes:  {summary['n_classes']}")
rows = []
for name in ['LogisticRegression', 'RandomForest', 'XGBoost']:
    if name in summary:
        for split in ['validation', 'test']:
            m = summary[name][split]
            rows.append({'Model': name, 'Split': split, **m})
df_metrics = pd.DataFrame(rows)
df_metrics.round(4)


## 3. Accuracy comparison chart

In [ ]:
pivot = df_metrics.pivot(index='Model', columns='Split', values='accuracy')
ax = pivot.plot(kind='bar', figsize=(9, 5), colormap='Set2', edgecolor='white')
ax.axhline(0.80, color='red', linestyle='--', label='80% target')
ax.set_title('Model Accuracy Comparison')
ax.set_ylabel('Accuracy'); ax.set_ylim(0.5, 1.0)
ax.legend(); plt.xticks(rotation=15); plt.tight_layout(); plt.show()


## 4. Confusion matrix (best model — test set)

In [ ]:
cm = pd.read_csv(ROOT / 'reports/confusion_matrix_best.csv', index_col=0)
fig, ax = plt.subplots(figsize=(18, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            linewidths=0.3, cbar=False)
ax.set_title('Confusion Matrix — Best Model (Test Set)')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()


## 5. Classification report

In [ ]:
print((ROOT / 'reports/classification_report_best.txt').read_text())
